# 2330 台股 L2 HftBacktest Smoke Test

這個 notebook 用 `data/tw_stock_events/2330_20250909.npz` 檢查兩件事：

1. 轉出的 HftBacktest event stream 是否能重建 BBO。
2. 若目前 Python kernel 已安裝可用的 `hftbacktest`，跑一個很小的 passive quote 策略，確認 backtest event loop / depth / 下單流程能走通。

目前 `qty` 單位保留為「張」，2330 在 2025-09-09 附近 tick size 使用 `5.0`，lot size 使用 `1.0`。

In [1]:
from pathlib import Path
import math
import sys

import numpy as np

ROOT = Path.cwd().parent
DATA_FILE = ROOT / "data" / "tw_stock_events" / "2330_20250909.npz"

TICK_SIZE = 5.0
LOT_SIZE = 1.0
CONTRACT_SIZE = 1000.0  # qty is in board lots; one board lot is 1000 shares.

DATA_FILE

WindowsPath('C:/Users/zoufuc/Desktop/hftbacktest/data/tw_stock_events/2330_20250909.npz')

In [2]:
data = np.load(DATA_FILE)["data"]

print("rows:", len(data))
print("dtype:", data.dtype)
print("first:", data[0])
print("last:", data[-1])
print("exch_ts range:", int(data["exch_ts"][0]), int(data["exch_ts"][-1]))
print("local-exch latency min/max:", int(np.min(data["local_ts"] - data["exch_ts"])), int(np.max(data["local_ts"] - data["exch_ts"])))

rows: 570212
dtype: [('ev', '<u8'), ('exch_ts', '<i8'), ('local_ts', '<i8'), ('px', '<f8'), ('qty', '<f8'), ('order_id', '<u8'), ('ival', '<i8'), ('fval', '<f8')]
first: (3758096387, 1757377804629985000, 1757377804629985000, 1165.0, 0.0, 0, 0, 0.0)
last: (3489660932, 1757395800000000000, 1757395800000000000, 1220.0, 1998.0, 0, 0, 0.0)
exch_ts range: 1757377804629985000 1757395800000000000
local-exch latency min/max: 0 0


In [3]:
DEPTH_EVENT = 1
TRADE_EVENT = 2
DEPTH_CLEAR_EVENT = 3
DEPTH_SNAPSHOT_EVENT = 4
EXCH_EVENT = 1 << 31
LOCAL_EVENT = 1 << 30
BUY_EVENT = 1 << 29
SELL_EVENT = 1 << 28
EVENT_FLAG_MASK = EXCH_EVENT | LOCAL_EVENT | BUY_EVENT | SELL_EVENT

def event_kind(ev):
    return int(ev) & ~EVENT_FLAG_MASK

kinds = np.array([event_kind(ev) for ev in data["ev"]])
print("depth events:", int(np.sum(np.isin(kinds, [DEPTH_EVENT, DEPTH_CLEAR_EVENT, DEPTH_SNAPSHOT_EVENT]))))
print("trade events:", int(np.sum(kinds == TRADE_EVENT)))
print("unique event kinds:", sorted(set(kinds.tolist())))

depth events: 562224
trade events: 7988
unique event kinds: [2, 3, 4]


In [4]:
# Lightweight replay: verify the event stream updates a top-of-book state.
# A single top-5 CSV row becomes multiple events with the same timestamp, so validate BBO after each timestamp batch.
bid_depth = {}
ask_depth = {}
samples = []
crossed = 0
last_ts = None

def check_batch(ts):
    global crossed
    if ts is None or not bid_depth or not ask_depth:
        return
    best_bid = max(bid_depth)
    best_ask = min(ask_depth)
    if best_bid >= best_ask:
        crossed += 1
    if len(samples) < 10:
        samples.append((int(ts), best_bid, best_ask, bid_depth[best_bid], ask_depth[best_ask]))

for row in data:
    ts = int(row["exch_ts"])
    if last_ts is not None and ts != last_ts:
        check_batch(last_ts)
    last_ts = ts

    ev = int(row["ev"])
    kind = event_kind(ev)
    px = float(row["px"])
    qty = float(row["qty"])

    if ev & BUY_EVENT:
        if kind == DEPTH_CLEAR_EVENT:
            for price in list(bid_depth):
                if price >= px:
                    del bid_depth[price]
        elif kind in (DEPTH_EVENT, DEPTH_SNAPSHOT_EVENT):
            if qty > 0:
                bid_depth[px] = qty
            else:
                bid_depth.pop(px, None)
    elif ev & SELL_EVENT:
        if kind == DEPTH_CLEAR_EVENT:
            for price in list(ask_depth):
                if price <= px:
                    del ask_depth[price]
        elif kind in (DEPTH_EVENT, DEPTH_SNAPSHOT_EVENT):
            if qty > 0:
                ask_depth[px] = qty
            else:
                ask_depth.pop(px, None)

check_batch(last_ts)

print("first BBO samples: exch_ts, bid, ask, bid_qty, ask_qty")
for item in samples:
    print(item)
print("crossed BBO count:", crossed)
assert len(samples) > 0
assert crossed == 0

first BBO samples: exch_ts, bid, ask, bid_qty, ask_qty
(1757377804629985000, 1190.0, 1195.0, 200.0, 3.0)
(1757377809646651000, 1195.0, 1200.0, 26.0, 148.0)
(1757377814662269000, 1190.0, 1195.0, 201.0, 5.0)
(1757377819677894000, 1195.0, 1200.0, 46.0, 276.0)
(1757377824693500000, 1190.0, 1195.0, 302.0, 46.0)
(1757377829710146000, 1190.0, 1195.0, 305.0, 66.0)
(1757377834727807000, 1190.0, 1195.0, 307.0, 81.0)
(1757377839743423000, 1190.0, 1195.0, 414.0, 6.0)
(1757377844759058000, 1190.0, 1195.0, 415.0, 17.0)
(1757377849772607000, 1190.0, 1195.0, 420.0, 118.0)
crossed BBO count: 0


## Optional: compare against DataAPI rows

這個 cell 會讀 `data_platform`，需要能連到 `\\DC_TW\taiwan_stock\數據平台`。如果只想檢查 `.npz`，可以跳過。

In [5]:
API_DIR = ROOT / "data_platform" / "data_stock" / "api"
if API_DIR.exists():
    sys.path.insert(0, str(API_DIR))
    from api_parquet import DataAPI

    api = DataAPI(base_dir="\\\\DC_TW\\taiwan_stock\\數據平台", index_backend="duckdb")
    df = api.get_data_single_symbol("2330", "2025-09-09", "2025-09-09")
    print(df.shape)
    print(df.head(3))
else:
    print("DataAPI directory not found; skipping optional comparison.")

2026-06-29 17:30:42,091 - INFO - Found 5 parquet files


2026-06-29 17:30:42,091 - INFO - Filtering for date range 20250909 to 20250909


(46852, 37)
shape: (3, 37)
┌───────────┬────────────┬────────────┬─────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ symbol_id ┆ exchtime   ┆ localtime  ┆ status  ┆ … ┆ ask_volum ┆ bid_price ┆ bid_volum ┆ sequence │
│ ---       ┆ ---        ┆ ---        ┆ ---     ┆   ┆ e5        ┆ 5         ┆ e5        ┆ ---      │
│ i64       ┆ i64        ┆ i64        ┆ i64     ┆   ┆ ---       ┆ ---       ┆ ---       ┆ i64      │
│           ┆            ┆            ┆         ┆   ┆ i64       ┆ i64       ┆ i64       ┆          │
╞═══════════╪════════════╪════════════╪═════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 1137      ┆ 1757377804 ┆ 1757377804 ┆ 8388826 ┆ … ┆ 22        ┆ 1165      ┆ 5         ┆ 719630   │
│           ┆ 629985000  ┆ 629985000  ┆         ┆   ┆           ┆           ┆           ┆          │
│ 1137      ┆ 1757377809 ┆ 1757377809 ┆ 8388826 ┆ … ┆ 94        ┆ 1175      ┆ 5         ┆ 720214   │
│           ┆ 646651000  ┆ 646651000  ┆         ┆   ┆           

## HftBacktest smoke strategy

???? notebook ?? `Python 3.11 (hftbacktest)` kernel??? kernelspec ???????? `hftbacktest 2.4.4` ? Python 3.11??? workspace ???? `hftbacktest/` repo ????? Python ?? shadow ???

In [6]:
import importlib
import site

print("kernel executable:", sys.executable)
print("python version:", sys.version)
print("site-packages:")
for package_path in site.getsitepackages():
    print(" ", package_path)

def import_hftbacktest_package(workspace_root: Path):
    root = workspace_root.resolve()
    original_path = list(sys.path)
    try:
        filtered_path = []
        for path_entry in original_path:
            if path_entry == "":
                continue
            try:
                if Path(path_entry).resolve() == root:
                    continue
            except Exception:
                pass
            filtered_path.append(path_entry)
        sys.path = filtered_path
        sys.modules.pop("hftbacktest", None)
        module = importlib.import_module("hftbacktest")
        if not hasattr(module, "BacktestAsset"):
            raise ImportError(f"imported {module!r} from {getattr(module, '__file__', None)} but BacktestAsset is missing")
        return module
    finally:
        sys.path = original_path

try:
    hbtpkg = import_hftbacktest_package(ROOT)
    print("loaded hftbacktest:", hbtpkg)
    print("package file:", getattr(hbtpkg, "__file__", None))
    print("version:", getattr(hbtpkg, "__version__", "unknown"))
except Exception as exc:
    hbtpkg = None
    print("hftbacktest package is not available in this kernel.")
    print("Use the notebook kernel named: Python 3.11 (hftbacktest)")
    print("Or install into this exact kernel with:")
    print(f"  {sys.executable} -m pip install hftbacktest")
    print("error:", repr(exc))

kernel executable: C:\Users\zoufuc\AppData\Local\Programs\Python\Python311\python.exe
python version: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
site-packages:
  C:\Users\zoufuc\AppData\Local\Programs\Python\Python311
  C:\Users\zoufuc\AppData\Local\Programs\Python\Python311\Lib\site-packages
loaded hftbacktest: <module 'hftbacktest' from 'C:\\Users\\zoufuc\\AppData\\Local\\Programs\\Python\\Python311\\Lib\\site-packages\\hftbacktest\\__init__.py'>
package file: C:\Users\zoufuc\AppData\Local\Programs\Python\Python311\Lib\site-packages\hftbacktest\__init__.py
version: 2.4.4


In [7]:
if hbtpkg is None:
    raise RuntimeError("hftbacktest package is not available; see previous cell.")

BacktestAsset = hbtpkg.BacktestAsset
HashMapMarketDepthBacktest = hbtpkg.HashMapMarketDepthBacktest

asset = (
    BacktestAsset()
    .data(str(DATA_FILE))
    .linear_asset(CONTRACT_SIZE)
    .constant_order_latency(1_000_000, 1_000_000)
    .risk_adverse_queue_model()
    .no_partial_fill_exchange()
    .trading_value_fee_model(0.0, 0.0)
    .tick_size(TICK_SIZE)
    .lot_size(LOT_SIZE)
    .last_trades_capacity(100)
)

hbt = HashMapMarketDepthBacktest([asset])
print("hbt built")

hbt built


In [8]:
from numba import njit

GTX = hbtpkg.GTX
LIMIT = hbtpkg.LIMIT

@njit
def passive_quote_smoke(hbt, max_steps):
    asset_no = 0
    submitted = False
    steps = 0
    first_bid = 0.0
    first_ask = 0.0

    while steps < max_steps:
        result = hbt.elapse(1_000_000_000)  # 1 second
        if result != 0:
            break

        depth = hbt.depth(asset_no)
        if np.isfinite(depth.best_bid) and np.isfinite(depth.best_ask):
            if first_bid == 0.0:
                first_bid = depth.best_bid
                first_ask = depth.best_ask

            if not submitted:
                bid_px = depth.best_bid - 2.0 * depth.tick_size
                ask_px = depth.best_ask + 2.0 * depth.tick_size
                hbt.submit_buy_order(asset_no, 1, bid_px, 1.0, GTX, LIMIT, False)
                hbt.submit_sell_order(asset_no, 2, ask_px, 1.0, GTX, LIMIT, False)
                submitted = True

        hbt.clear_inactive_orders(asset_no)
        steps += 1

    state = hbt.state_values(asset_no)
    return steps, first_bid, first_ask, hbt.position(asset_no), state.num_trades, state.trading_value

result = passive_quote_smoke(hbt, 120)
print("steps, first_bid, first_ask, position, num_trades, trading_value =")
print(result)

assert result[0] > 0
assert result[1] > 0
assert result[2] > result[1]

hbt.close()
print("smoke backtest passed")

steps, first_bid, first_ask, position, num_trades, trading_value =
(120, 1190.0, 1195.0, 0.0, 0, 0.0)
smoke backtest passed


## Aggressive fill state report

??????? backtest instance??? `GTC LIMIT` ???? best ask??? best bid?????????????? BBO?mark price?position?balance?fee?equity?num_trades?trading_value?trading_volume?

In [9]:
FILL_MAKER_FEE = 0.0
FILL_TAKER_FEE = 0.0
FILL_ORDER_LATENCY_NS = 0
FILL_QTY = 1.0


def build_aggressive_fill_backtest():
    asset = (
        hbtpkg.BacktestAsset()
        .data(str(DATA_FILE))
        .linear_asset(CONTRACT_SIZE)
        .constant_order_latency(FILL_ORDER_LATENCY_NS, FILL_ORDER_LATENCY_NS)
        .risk_adverse_queue_model()
        .no_partial_fill_exchange()
        .trading_value_fee_model(FILL_MAKER_FEE, FILL_TAKER_FEE)
        .tick_size(TICK_SIZE)
        .lot_size(LOT_SIZE)
        .last_trades_capacity(100)
    )
    return hbtpkg.HashMapMarketDepthBacktest([asset])


def state_snapshot_for_report(hbt, asset_no=0):
    depth = hbt.depth(asset_no)
    state = hbt.state_values(asset_no)
    best_bid = float(depth.best_bid)
    best_ask = float(depth.best_ask)
    if np.isfinite(best_bid) and np.isfinite(best_ask):
        mark_px = (best_bid + best_ask) / 2.0
    elif np.isfinite(best_bid):
        mark_px = best_bid
    elif np.isfinite(best_ask):
        mark_px = best_ask
    else:
        mark_px = 0.0

    position = float(state.position)
    balance = float(state.balance)
    fee = float(state.fee)
    equity = balance + position * mark_px * CONTRACT_SIZE - fee
    return {
        "best_bid": best_bid,
        "best_ask": best_ask,
        "mark_px": mark_px,
        "position": position,
        "balance": balance,
        "fee": fee,
        "equity": equity,
        "num_trades": int(state.num_trades),
        "trading_value": float(state.trading_value),
        "trading_volume": float(state.trading_volume),
    }


def print_state_report(label, order_id, snap):
    order_text = "" if order_id is None else f" order_id={order_id}"
    print(
        f"{label:<18}{order_text:<14}"
        f" bid={snap['best_bid']:.2f}"
        f" ask={snap['best_ask']:.2f}"
        f" mark={snap['mark_px']:.2f}"
        f" pos={snap['position']:.4f}"
        f" balance={snap['balance']:.2f}"
        f" fee={snap['fee']:.2f}"
        f" equity={snap['equity']:.2f}"
        f" trades={snap['num_trades']}"
        f" value={snap['trading_value']:.2f}"
        f" volume={snap['trading_volume']:.4f}"
    )


def wait_for_valid_bbo(hbt, asset_no=0, step_ns=1_000_000_000, max_steps=10):
    for _ in range(max_steps):
        result = hbt.elapse(step_ns)
        if result != 0:
            raise RuntimeError("Backtest ended before a valid BBO was available.")
        depth = hbt.depth(asset_no)
        if np.isfinite(depth.best_bid) and np.isfinite(depth.best_ask):
            return
    raise RuntimeError("No valid BBO found during warmup.")


def submit_crossing_order_and_report(hbt, side, order_id, px, qty=FILL_QTY, asset_no=0):
    before_trades = state_snapshot_for_report(hbt, asset_no)["num_trades"]
    print_state_report(f"before_{side}", order_id, state_snapshot_for_report(hbt, asset_no))

    if side == "buy":
        rc = hbt.submit_buy_order(asset_no, order_id, px, qty, hbtpkg.GTC, hbtpkg.LIMIT, False)
    elif side == "sell":
        rc = hbt.submit_sell_order(asset_no, order_id, px, qty, hbtpkg.GTC, hbtpkg.LIMIT, False)
    else:
        raise ValueError(side)
    print(f"submit_{side:<11} order_id={order_id:<6} px={px:.2f} qty={qty:.4f} rc={rc}")

    response = hbt.wait_order_response(asset_no, order_id, 10_000_000)
    after = state_snapshot_for_report(hbt, asset_no)
    print(f"response_{side:<9} order_id={order_id:<6} response={response}")
    print_state_report(f"after_{side}", order_id, after)

    assert after["num_trades"] == before_trades + 1, f"{side} did not fill"
    hbt.clear_inactive_orders(asset_no)
    print_state_report(f"clear_{side}", order_id, state_snapshot_for_report(hbt, asset_no))


def run_aggressive_fill_state_report(round_trips=1):
    if hbtpkg is None:
        raise RuntimeError("hftbacktest package is not available; run the import cell first.")

    hbt_fill = build_aggressive_fill_backtest()
    asset_no = 0
    next_order_id = 20_001
    try:
        wait_for_valid_bbo(hbt_fill, asset_no)
        print_state_report("initial_bbo", None, state_snapshot_for_report(hbt_fill, asset_no))

        for round_no in range(1, round_trips + 1):
            depth = hbt_fill.depth(asset_no)
            print(f"\nround={round_no} aggressive buy at best ask")
            submit_crossing_order_and_report(
                hbt_fill,
                "buy",
                next_order_id,
                float(depth.best_ask),
                FILL_QTY,
                asset_no,
            )
            next_order_id += 1

            depth = hbt_fill.depth(asset_no)
            print(f"\nround={round_no} aggressive sell at best bid")
            submit_crossing_order_and_report(
                hbt_fill,
                "sell",
                next_order_id,
                float(depth.best_bid),
                FILL_QTY,
                asset_no,
            )
            next_order_id += 1

        print("\nfinal")
        final_state = state_snapshot_for_report(hbt_fill, asset_no)
        print_state_report("final_state", None, final_state)
        assert final_state["position"] == 0.0
        return final_state
    finally:
        hbt_fill.close()


aggressive_final_state = run_aggressive_fill_state_report(round_trips=1)
aggressive_final_state

initial_bbo                      bid=1190.00 ask=1195.00 mark=1192.50 pos=0.0000 balance=0.00 fee=0.00 equity=0.00 trades=0 value=0.00 volume=0.0000

round=1 aggressive buy at best ask
before_buy         order_id=20001 bid=1190.00 ask=1195.00 mark=1192.50 pos=0.0000 balance=0.00 fee=0.00 equity=0.00 trades=0 value=0.00 volume=0.0000
submit_buy         order_id=20001  px=1195.00 qty=1.0000 rc=0
response_buy       order_id=20001  response=0
after_buy          order_id=20001 bid=1190.00 ask=1195.00 mark=1192.50 pos=1.0000 balance=-1195000.00 fee=0.00 equity=-2500.00 trades=1 value=1195000.00 volume=1.0000
clear_buy          order_id=20001 bid=1190.00 ask=1195.00 mark=1192.50 pos=1.0000 balance=-1195000.00 fee=0.00 equity=-2500.00 trades=1 value=1195000.00 volume=1.0000

round=1 aggressive sell at best bid
before_sell        order_id=20002 bid=1190.00 ask=1195.00 mark=1192.50 pos=1.0000 balance=-1195000.00 fee=0.00 equity=-2500.00 trades=1 value=1195000.00 volume=1.0000
submit_sell        

{'best_bid': 1190.0,
 'best_ask': 1195.0,
 'mark_px': 1192.5,
 'position': 0.0,
 'balance': -5000.0,
 'fee': 0.0,
 'equity': -5000.0,
 'num_trades': 2,
 'trading_value': 2385000.0,
 'trading_volume': 2.0}